In [ ]:
# 目的：打印本章依赖库的版本，确认环境就绪（版本差异可能影响权重加载/生成结果）。
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 目的：单独检查 TensorFlow 与 tqdm 版本。
# 说明：本章要读取 OpenAI 官方 GPT-2 的 TensorFlow checkpoint，因此必须安装 TensorFlow。
#       这也印证了 gpt_download.py 里应当 `import tensorflow as tf`（而非误写的 transformers）。
print("TensorFlow version:", version("tensorflow"))
print("tqdm version:", version("tqdm"))

In [ ]:
# 目的：从同目录的 gpt_download.py 导入“下载并加载官方 GPT-2 权重”的主函数。
# Relative import from the gpt_download.py contained in this folder
from gpt_download import download_and_load_gpt2

In [ ]:
# 目的：下载并加载 124M 规模的 GPT-2 权重，保存到本地 "gpt2" 目录（已存在且大小一致则跳过下载）。
# 返回：settings 为超参数字典；params 为解析好的嵌套权重字典（含各层 blocks 列表）。
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
# 目的：查看从 hparams.json 读到的模型超参数（如 n_layer/n_head/n_embd/n_ctx/n_vocab）。
print("Settings:", settings)

In [ ]:
# 目的：查看权重字典的顶层键，并以 token 嵌入 wte 为例看其数值与形状。
# wte 形状约为 (50257, 768)，即“词表大小 x 嵌入维度”。
print("Parameter dictionary keys:", params.keys())
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

In [ ]:
# 目的：定义 GPT-2 各规模的配置，并从 124M 基础配置派生出用于“加载官方权重”的 NEW_CONFIG。
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}


# 不同规模的 GPT-2 仅在 emb_dim / n_layers / n_heads 上不同，其余超参共用。
# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
# 复制基础配置，再用所选规模的设置覆盖，得到本次要构建的模型配置。
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
# 关键对齐：把上下文长度改回原版 1024，并启用 qkv_bias=True。
# 因为官方 GPT-2 的注意力权重是“带 QKV 偏置”的，且训练时上下文长度为 1024，
# 必须与官方一致，否则下一步 assign() 会因形状不匹配而报错。
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

In [ ]:
# 目的：用 NEW_CONFIG 实例化我们自建的 GPTModel，并切到 eval 模式（关闭 dropout）。
# 注意：此刻权重仍是随机初始化，下一步才会把官方权重灌进去。
from supplementary import GPTModel

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [ ]:
# 目的：定义权重赋值辅助函数 assign。
# 作用：把 right（numpy 数组）搬进 left（模型参数）前先做形状校验，
#       形状不一致立即抛错，防止把权重错配到不匹配的层上。
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))  # 用 numpy 数值新建一个可训练参数返回

In [ ]:
# 目的（本章核心）：把 OpenAI 官方 GPT-2 的权重逐一搬进我们自建的 GPTModel。
# 三个关键点：
#   1) c_attn 把 Q/K/V 三套权重“合并”存在一起，需用 np.split 沿最后一维切成三份；
#   2) 权重需要转置 (.T)：TF/官方用的是 x@W 约定，而 PyTorch 的 nn.Linear 存的是 W 的转置；
#   3) out_head 与 tok_emb 共享同一份权重（weight tying），故最后用 params["wte"] 赋给 out_head。
import torch
import numpy as np

def load_weights_into_gpt(gpt, params):
    # 位置嵌入 wpe 与 token 嵌入 wte（这两者本身无需转置）。
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    # 逐个 Transformer 块搬运权重。
    for b in range(len(params["blocks"])):
        # c_attn 里 Q/K/V 权重按最后一维拼在一起（宽度 3*emb_dim），切成三份。
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        # 每份都要 .T 转置后再赋给对应的 W_query/W_key/W_value。
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        # QKV 的偏置同样是合并存放，按最后一维切成三份（偏置是一维，无需转置）。
        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        # 注意力输出投影 c_proj（权重转置，偏置直接搬）。
        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        # 前馈网络第 1 层 c_fc（升维，layers[0]）：权重转置、偏置直接搬。
        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        # 前馈网络第 2 层 c_proj（降维，layers[2]；layers[1] 是 GELU 无参数）。
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        # 两处 LayerNorm 参数：g->scale(gamma)、b->shift(beta)。ln_1 对应 norm1，ln_2 对应 norm2。
        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    # 末层归一化的全局 g/b。
    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    # weight tying：输出头直接复用 token 嵌入权重 wte（输入嵌入与输出投影共享同一张表）。
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])


# 执行搬运：调用后 gpt 即拥有官方预训练权重。
load_weights_into_gpt(gpt, params)

# 选择计算设备（有 GPU 用 cuda，否则 cpu）并把模型迁移过去。
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt.to(device);

In [ ]:
# 目的：用加载好官方权重的模型做一次文本生成，验证权重搬运是否正确。
# 判断标准：若能续写出通顺英文（而非乱码），说明权重加载成功。
import tiktoken
from supplementary import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)


tokenizer = tiktoken.get_encoding("gpt2")

torch.manual_seed(123)  # 固定随机种子（贪心生成本确定，此处为规范做法，便于复现）

token_ids = generate_text_simple(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=10,
    # 注意：这里传的是 GPT_CONFIG_124M["context_length"]（=256），
    #       而模型实际按 NEW_CONFIG 以 1024 构建。生成时会把上下文裁剪到 256；
    #       本例输入很短不受影响，若要用满 1024 上下文应改传 NEW_CONFIG。
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
#Trying larger LLMs
# 提示：要加载更大规模（355M/774M/1558M），只需把 model_size 与所选的
#       model_configs 项换成对应规模，权重加载流程完全一致
#       （因为 gpt_download 解析出的 params 结构对所有规模都相同）。